# Notebook 04 — Training

**Goal:** Fine-tune `facebook/wav2vec2-large-xlsr-53-arabic` on the aligned
Qaloon dataset from Notebook 03 and save the best checkpoint.

**Why this notebook exists:**
This is the core machine learning step.  We take the pre-trained multilingual
Wav2Vec2 model (which already understands Arabic phonemes from large-scale
pre-training) and fine-tune its transformer layers plus a new CTC head on our
Qaloon-specific data.  The CNN feature extractor is frozen because it extracts
low-level acoustic features that don't need to change.

**Runtime:** ~4–6 hours on A10G, ~10–14 hours on T4.

**Output:** `models/qaloon_wav2vec2/` — HuggingFace model checkpoint

## Step 0 — Setup and GPU check

We check GPU availability first.  Training on CPU is not practical for this
model size — if no GPU is detected, switch to a GPU runtime before continuing.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU detected. Switch to a GPU runtime.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/drive/MyDrive/iqraa-ai')
!pip install -q -r /content/drive/MyDrive/iqraa-ai/requirements.txt

## Step 1 — Configure paths

In [ ]:
from pathlib import Path

REPO_ROOT   = Path('/content/drive/MyDrive/iqraa-ai')
CONFIG_FILE = REPO_ROOT / 'configs' / 'training_config.yaml'
VOCAB_FILE  = REPO_ROOT / 'data' / 'processed' / 'vocab.json'
DATASET_DIR = REPO_ROOT / 'data' / 'processed' / 'hf_dataset'
MODEL_OUT   = REPO_ROOT / 'models' / 'qaloon_wav2vec2'

MODEL_OUT.mkdir(parents=True, exist_ok=True)

## Step 2 — Load dataset and build processor

The processor wraps the feature extractor (audio → float array) and tokenizer
(text → token IDs) into a single object that handles both directions.

In [ ]:
from src.dataset import load_dataset_from_disk
from src.train import build_processor, preprocess_dataset, load_config

cfg = load_config(str(CONFIG_FILE))
ds  = load_dataset_from_disk(str(DATASET_DIR))
processor = build_processor(str(VOCAB_FILE), cfg['model']['base_model_id'])

print(ds)
print(f'Vocabulary size: {len(processor.tokenizer)}')

## Step 3 — Preprocess audio and text for the model

We extract raw input values from the audio arrays and convert text labels to
token ID sequences.  This is done once here and cached so training doesn't
repeat the feature extraction on every step.

In [ ]:
ds_processed = preprocess_dataset(ds, processor)
print(ds_processed)

## Step 4 — Load base model and freeze feature extractor

We load from the HuggingFace Hub the first time (downloads ~1.2 GB), then
freeze the CNN layers.  Only the transformer blocks and the new CTC projection
head are trained — this reduces memory and speeds convergence.

In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    cfg['model']['base_model_id'],
    ctc_loss_reduction=cfg['ctc']['ctc_loss_reduction'],
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

if cfg['model']['freeze_feature_extractor']:
    model.freeze_feature_extractor()
    print('Feature extractor frozen.')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({trainable/total:.1%})')

## Step 5 — Train

All hyperparameters come from `configs/training_config.yaml`.  The Trainer
saves checkpoints every `eval_steps` and keeps the best model by WER.
EarlyStopping will halt training if WER stops improving for 3 evaluations.

In [ ]:
from src.train import DataCollatorCTCWithPadding
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=str(MODEL_OUT),
    num_train_epochs=cfg['training']['num_train_epochs'],
    per_device_train_batch_size=cfg['training']['per_device_train_batch_size'],
    per_device_eval_batch_size=cfg['training']['per_device_eval_batch_size'],
    gradient_accumulation_steps=cfg['training']['gradient_accumulation_steps'],
    learning_rate=cfg['training']['learning_rate'],
    warmup_steps=cfg['training']['warmup_steps'],
    weight_decay=cfg['training']['weight_decay'],
    fp16=cfg['training']['fp16'],
    evaluation_strategy=cfg['training']['evaluation_strategy'],
    eval_steps=cfg['training']['eval_steps'],
    save_steps=cfg['training']['save_steps'],
    logging_steps=cfg['training']['logging_steps'],
    load_best_model_at_end=cfg['training']['load_best_model_at_end'],
    metric_for_best_model=cfg['training']['metric_for_best_model'],
    greater_is_better=cfg['training']['greater_is_better'],
)

collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_processed['train'],
    eval_dataset=ds_processed['test'],
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

## Step 6 — Save the final model and processor

Saving the processor alongside the model ensures we can reload both with a
single `from_pretrained()` call in the evaluation and inference notebooks.

In [ ]:
model.save_pretrained(str(MODEL_OUT))
processor.save_pretrained(str(MODEL_OUT))
print(f'Model saved to {MODEL_OUT}')